In [2]:
import os
import pandas as pd
import numpy as np
import scipy.io as sio


In [3]:
GOLFDB_DIR = "/Users/austinblee/github/golfdb"
GOLFDB_MAT = f"{GOLFDB_DIR}/data/golfDB.mat"

print("GOLFDB_DIR:", GOLFDB_DIR)
print("GOLFDB_MAT:", GOLFDB_MAT)

/Users/austinblee/github/capstone/golfdb
['util.py', 'test_video.py', 'model.py', 'README.md', 'MobileNetV2.py', '.gitignore', 'train.py', 'dataloader.py', 'eval.py', '.git', 'data', 'test_video.mp4']


In [4]:
DATA_DIR = "data"

print(os.listdir(DATA_DIR))

['preprocess_videos.py', 'golfDB.pkl', 'generate_splits.py', 'golfDB.mat']


In [5]:
mat = sio.loadmat(GOLFDB_MAT)
print(mat.keys())

golfdb = mat["golfDB"]

print(type(golfdb))
print(golfdb.shape)
print(golfdb.dtype)
print(golfdb.dtype.names)

dict_keys(['__header__', '__version__', '__globals__', 'golfDB'])
<class 'numpy.ndarray'>
(1, 1400)
[('id', 'O'), ('youtube_id', 'O'), ('player', 'O'), ('sex', 'O'), ('club', 'O'), ('view', 'O'), ('slow', 'O'), ('events', 'O'), ('bbox', 'O'), ('split', 'O')]
('id', 'youtube_id', 'player', 'sex', 'club', 'view', 'slow', 'events', 'bbox', 'split')


In [12]:

def unwrap(x):
    """
    Makes MATLAB-loaded arrays easier to inspect.
    """
    while isinstance(x, np.ndarray) and x.size == 1:
        x = x.item()
    return x


# first three records
for i in range(3):
    record = golfdb[0, i]

    print(f"\n========== RECORD {i} ==========")

    for field in golfdb.dtype.names:
        value = unwrap(record[field])
        print(f"{field}: {value}")


========== RECORD 0 ==========
id: 0
youtube_id: f1BWA5F87Jc
player: SANDRA GAL
sex: f
club: driver
view: down-the-line
slow: 0
events: [[408 455 473 476 490 495 498 501 514 545]]
bbox: [[0.09765625 0.00694444 0.50234375 0.98055556]]
split: 3

========== RECORD 1 ==========
id: 1
youtube_id: f1BWA5F87Jc
player: SANDRA GAL
sex: f
club: driver
view: down-the-line
slow: 1
events: [[ 814  854  917  931  988 1006 1019 1030 1083 1137]]
bbox: [[3.90625000e-02 6.94444444e-04 6.12500000e-01 9.78472222e-01]]
split: 3

========== RECORD 2 ==========
id: 2
youtube_id: tA1iotgtMyc
player: CHRIS DIMARCO
sex: m
club: driver
view: down-the-line
slow: 0
events: [[521 659 678 683 692 696 698 701 715 745]]
bbox: [[1.65625000e-01 6.94444444e-04 4.83593750e-01 9.86805556e-01]]
split: 3


In [14]:
#set up df
rows = [] 
for i in range(golfdb.shape[1]):
    record = golfdb[0, i]
    row = {}
    for field in golfdb.dtype.names:
        row[field] = unwrap(record[field])
    rows.append(row)
#create df    
df = pd.DataFrame(rows)

df.head()

,id,youtube_id,player,sex,club,view,slow,events,bbox,split
0,0,f1BWA5F87Jc,SANDRA GAL,f,driver,down-the-line,0,"[[408, 455, 473, 476, 490, 495, 498, 501, 514,...","[[0.09765625000000001, 0.006944444444444444, 0...",3
1,1,f1BWA5F87Jc,SANDRA GAL,f,driver,down-the-line,1,"[[814, 854, 917, 931, 988, 1006, 1019, 1030, 1...","[[0.039062500000000014, 0.0006944444444444445,...",3
2,2,tA1iotgtMyc,CHRIS DIMARCO,m,driver,down-the-line,0,"[[521, 659, 678, 683, 692, 696, 698, 701, 715,...","[[0.165625, 0.0006944444444444445, 0.48359375,...",3
3,3,tA1iotgtMyc,CHRIS DIMARCO,m,driver,down-the-line,1,"[[1106, 1190, 1244, 1264, 1300, 1313, 1324, 13...","[[0.18515625, 0.0006944444444444445, 0.465625,...",3
4,4,wDCKLePrwHA,BROOKE HENDERSON,f,driver,down-the-line,0,"[[157, 170, 183, 188, 197, 201, 205, 207, 220,...","[[0.11015625, 0.0006944444444444445, 0.4984375...",3


In [16]:
print(df.columns)

# How many videos?
print("Number of records:", len(df))

print("\nViews:")
if "view" in df.columns:
    print(df["view"].value_counts())
else:
    print("no view column")
    
print("\nClubs:")
if "club" in df.columns:
    print(df["club"].value_counts())
else:
    print("no club column")

print("\nPlayers:")
if "player" in df.columns:
    print(df["player"].value_counts().head(20))
else:
    print("no player column")

Index(['id', 'youtube_id', 'player', 'sex', 'club', 'view', 'slow', 'events',
       'bbox', 'split'],
      dtype='object')
Number of records: 1400

Views:
view
down-the-line    585
face-on          461
other            354
Name: count, dtype: int64

Clubs:
club
driver     952
iron       229
fairway    162
hybrid      34
wedge       23
Name: count, dtype: int64

Players:
player
LYDIA KO            58
MICHELLE WIE        54
TIGER WOODS         53
BROOKE HENDERSON    35
RORY MCILROY        27
INBEE PARK          23
LEXI THOMPSON       23
PAULA CREAMER       21
GRAEME MCDOWELL     19
RICKIE FOWLER       18
CHARLEY HULL        17
BERNHARD LANGER     17
ADAM SCOTT          17
SANDRA GAL          16
JUSTIN ROSE         16
SEAN OHAIR          15
HENRIK STENSON      15
BUBBA WATSON        14
FRED COUPLES        14
STACY LEWIS         13
Name: count, dtype: int64
